In [9]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from spectral import envi # Para os arquivos .bil/.hdr
from math import pi
from skimage import morphology
import json
import os
import glob

# Funções do Pipeline

In [10]:
def prepare_img(img):
  mask = np.ones_like(img, dtype=bool)
  mask[30:600, 200:650] = False

  copy = img.copy()
  copy[mask] = 0
  return copy

## Funções de Visualização

In [11]:
def plot_homography(img_rgb, homographed_rx):
  crop = prepare_img(img_rgb)
  blended = cv2.addWeighted(crop, 0.5, homographed_rx, 0.5, 0)

  plt.figure(figsize=(10,10))
  plt.imshow(cv2.cvtColor(blended, cv2.COLOR_BGR2RGB))
  plt.title("Sobreposição após Homografia")
  plt.axis('off')
  plt.show()

In [12]:
def debug_centroids(img, grade, contours = True, num_sementes = 50):
  copy = img.copy()
  copy = cv2.cvtColor(copy, cv2.COLOR_BGR2RGB)
  for i in range(num_sementes):
    point = grade[i]
    point = (int(point[0]), int(point[1]))
    cv2.circle(copy, point, radius = 0, color=(255, 255, 255), thickness = 5)
    #cv2.putText(copy, f"{point[0]}, {point[1]}", (point[0] +5, point[1] +5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)
    if contours:
      cv2.drawContours(copy, grade[i][1], -1, (0, 255, 0), 1)
  plt.axis('off')
  plt.imshow(copy)

In [13]:
def plot_segmentation(img, contours, centroids = None, rgb = True):
  if rgb:
    crop = prepare_img(img)
  else:
    crop = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
  if centroids:
    for point in centroids:
      cv2.circle(crop, point, 5, (255, 255, 255), -1)

  for i, cnt in enumerate(contours):
    x, y, w, h = cv2.boundingRect(cnt)
    cv2.rectangle(crop, (x, y), (x + w, y + h), (255, 0, 0), 1)
    cv2.putText(crop, str(i), (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)

  cv2.drawContours(crop, contours, -1, (0, 255, 0), 1)

  plt.figure(figsize=(12, 10))
  plt.imshow(crop)
  plt.title(f"Foram detectados {len(contours)} objetos")
  plt.show()
  plt.close()

## Funções de Segmentação

In [39]:
def calculate_morphology(erosion_disk, dilation_disk, img):
  #threshold de otsu
  _, thresh = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

  #erosao (para tirar ruido)
  disk = morphology.disk(erosion_disk)
  eroded = morphology.erosion(thresh, disk)

  #dilatacao (para aumentar as sementes encontradas)
  disk = morphology.disk(dilation_disk)
  eroded_dilated = morphology.dilation(eroded, disk)

  #convertendo em int, pois o morphology retorna um array binario
  converted = np.array(eroded_dilated, dtype=int)

  #normalizando de 0 e 1s para 0 e 255
  converted = cv2.normalize(converted, None, 0, 255, cv2.NORM_MINMAX, cv2.CV_8U)

  return converted

In [40]:
def segmentation_rx(img, rotate = True):
  #rotacao para deixar na mesma orientacao q o rgb
  if rotate:
    copy = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
  else:
    copy = img.copy()
  gray = cv2.cvtColor(copy, cv2.COLOR_BGR2GRAY)
  #_, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
  result = calculate_morphology(5, 2, gray)
  contours, _ = cv2.findContours(result, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

  filted_contours = []
  #em todos os contornos encontrados:
  for cnt in contours:
    if cv2.contourArea(cnt)>0.3: #filtragem de ruidos
          filted_contours.append(cnt)

  return tuple(filted_contours)

In [41]:
def segmentation_rgb(img):
  #cortando imagem
  crop = prepare_img(img)

  #transformacao em HSV
  hsv_img = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)

  #pegando apenas a informacao de saturacao
  saturation = hsv_img[:,:,1] #as sementes sao saturadas em relacao ao fundo

  result = calculate_morphology(13, 3, saturation)

  #encontrando contornos com o opencv
  contours, _ = cv2.findContours(result, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

  filtred_contours = []
  #em todos os contornos encontrados:
  for cnt in contours:
    #calculando a circularidade
    circ = (4*pi*cv2.contourArea(cnt))/(cv2.arcLength(cnt,True))**2
    #filtrando ruidos
    if circ>0.3:
      filtred_contours.append(cnt)

  return tuple(filtred_contours)

In [42]:
def calculate_centroids(contours):
  centroids = []
  for c in contours:
    M = cv2.moments(c)

    # calculando as coordenadas x,y do centro
    cX = int(M["m10"] / M["m00"])
    cY = int(M["m01"] / M["m00"])

    centroids.append([cX, cY])
  return centroids

In [43]:
def calculate_corner_points(centroids):
  s = [x + y for [x, y] in centroids]
  diff = [x - y for [x, y] in centroids]

  up_left = centroids[s.index(min(s))]
  bottom_right =centroids[s.index(max(s))]
  up_right = centroids[diff.index(max(diff))]
  bottom_left = centroids[diff.index(min(diff))]

  return np.array([up_right, bottom_right, bottom_left, up_left], dtype='float32')

In [44]:
def ordenate_points(points, contours = None):
  #ordenando primeiramente em relacao a y (de baixo para cima)
  if contours is None:
    y_ordenated = sorted(points, key = lambda point: point[0][1])
  else:
    y_ordenated = sorted(zip(points,contours), key = lambda point: point[0][1])
  ordenated = []
  rows = len(y_ordenated)//5

  i = 0
  for _ in range(rows):
    x_ordenated = y_ordenated[i:i+5]
    #ordenando de 5 em 5 em relacao a x (esq para a direita)
    x_ordenated = sorted(x_ordenated, key=lambda point: point[0][0])
    for j in range(5):
      ordenated.append(x_ordenated[j])
    i = i+5

  if contours is None:
    return ordenated
  else:
    cents, conts = zip(*ordenated)
    return cents, conts
    
  

In [45]:
def anote_points(data):
  points = []
  for shape in data['shapes']:
    points.append([shape['points'][0], shape['label']])
  return points

In [46]:
def homography(img_rx, centroids_rx, img_rgb, centroids_rgb):
  c_rgb = np.array(centroids_rgb)
  c_rx = np.array(centroids_rx)
  M, status = cv2.findHomography(c_rx, c_rgb, cv2.RANSAC, 1.0)

  copy_rx = cv2.rotate(img_rx, cv2.ROTATE_90_CLOCKWISE)

  h_rx, w_rx = img_rgb.shape[:2]
  homographed_rx = cv2.warpPerspective(copy_rx, M, (w_rx, h_rx))

  return homographed_rx

In [47]:
def crop_seeds_centered(img, centroids, points, sizeX, sizeY, hiper =None):
  cropped_img = []
  cropped_hiper = []
  H, W = img.shape[:2]
  halfX = sizeX // 2
  halfY = sizeY // 2

  if points is not None:
    points_ordenated = ordenate_points(points)
  for i, cent in enumerate(centroids):
    x0 = int(cent[0]-halfX)
    x1 = int(cent[0]+halfX)

    y0 = int(cent[1]-halfY)
    y1 = int(cent[1]+halfY)

    seed = img[y0:y1, x0:x1]
    if points is not None:
      seed = [seed, points_ordenated[i][1]]
    else:
      seed = [seed,""]
      
    cropped_img.append(seed)

    if hiper is not None:
      hp = hiper[:, y0:y1, x0:x1]
      cropped_hiper.append(hp)
  return cropped_img, cropped_hiper

# Classe do Pipeline

In [48]:
class SegmentationPipeline:
  def __init__(self,img):
    self.rgb_img = img["rgb"]
    self.rx_img = img["rx"]
    self.hiper_imgs = img["hiper"]
    self.json = img["json"]
    self.SIZEX = 64
    self.SIZEY = 45

  def run(self):
    if self.json is not None:
        points = anote_points(self.json)
    else:
        points = None
    contours = segmentation_rgb(self.rgb_img)
    centroids = calculate_centroids(contours)
    ordenated_centroids_rgb, _ = ordenate_points(centroids,contours)
    

    contours_rx = segmentation_rx(self.rx_img, rotate=True)
    centroids_rx = calculate_centroids(contours_rx)

    ordenated_centroids_rx, _ = ordenate_points(centroids_rx, contours_rx)

    new_rx = homography(self.rx_img, ordenated_centroids_rx, self.rgb_img, ordenated_centroids_rgb)

    new_cnt_rx = segmentation_rx(new_rx, rotate = False)
    new_cen_rx = calculate_centroids(new_cnt_rx)
    ocen, _ = ordenate_points(new_cen_rx,new_cnt_rx)

    self.cropped_rgb, self.cropped_hiper = crop_seeds_centered(self.rgb_img,
                                                               ordenated_centroids_rgb,
                                                               points,
                                                               self.SIZEX,self.SIZEY,
                                                               hiper = self.hiper_imgs)
    self.cropped_rx, _ = crop_seeds_centered(new_rx, ocen, points, self.SIZEX,self.SIZEY)

  def plot_seed(self, n_seed, n_band):
      fig, axs = plt.subplots(1, 4, figsize=(12, 4))

      rgb = cv2.cvtColor(self.cropped_rgb[n_seed][0], cv2.COLOR_BGR2RGB)
      axs[0].imshow(rgb)
      axs[0].set_title('RGB')
      axs[0].axis('off')

      axs[1].imshow(self.cropped_rx[n_seed])
      axs[1].set_title('RX')
      axs[1].axis('off')

      axs[2].imshow(self.cropped_hiper[n_seed][n_band])
      axs[2].set_title(f'Hiperspectral - Banda {n_band}')
      axs[2].axis('off')

      blended = cv2.addWeighted(rgb, 0.5, self.cropped_rx[n_seed], 0.5, 0)
      axs[3].imshow(blended)
      axs[3].set_title(f'RGB + RX')
      axs[3].axis('off')

      plt.tight_layout()
      plt.show()
    
  def save_cropped_seeds(self, output_dir, image_id):
      #cria as pastas de destino se elas não existirem
      rgb_dir = os.path.join(output_dir, "RGB")
      rx_dir = os.path.join(output_dir, "RX")
      hiper_dir = os.path.join(output_dir, "HIPER")  
      
      os.makedirs(rgb_dir, exist_ok=True)
      os.makedirs(rx_dir, exist_ok=True)
      os.makedirs(hiper_dir, exist_ok=True)
      
      num_seeds = len(self.cropped_rgb)
      
      for n_seed in range(num_seeds):
          filename = f"{image_id}_seed_{n_seed:02d}_{self.cropped_rgb[n_seed][1]}.png"
          img_rgb = self.cropped_rgb[n_seed][0] 
          
          if isinstance(img_rgb, np.ndarray):
            path_rgb = os.path.join(rgb_dir, filename)
            cv2.imwrite(path_rgb, img_rgb)
          else:
              print(f"  [Aviso] Objeto RGB inválido na semente {n_seed} da {image_id}")

          img_rx = self.cropped_rx[n_seed][0]

          if isinstance(img_rx, np.ndarray):
            path_rx = os.path.join(rx_dir, filename)
            cv2.imwrite(path_rx, img_rx)
          else:
              print(f"  [Aviso] Objeto RX inválido na semente {n_seed} da {image_id}")

          if self.cropped_hiper is not None and n_seed < len(self.cropped_hiper):
                img_hiper = self.cropped_hiper[n_seed]
                path_hiper = os.path.join(hiper_dir, f"{image_id}_seed_{n_seed:02d}.npy")
                np.save(path_hiper, img_hiper)

      print(f"-> {num_seeds} sementes da {image_id} salvas com sucesso em '{output_dir}'!")


# Segmentação

In [ ]:
path = ""
dados_anotados = ["011", "066", "098", "129", "133", "225", "277", "351", "531", "567", "598"]

DIRETORIO_SAIDA = ""

In [49]:
def read_seeds(array_seeds, is_labeled = True):
    for _, data in enumerate(array_seeds):
        rgb_format = f"{path}/RIP2_{data}-RGB.tif"
        bil_format = f"{path}/RIP2_{data}.bil"
        hdr_format = f"{path}/RIP2_{data}.bil.hdr"
        rx_format = f"{path}/RIP2_{data}.bmp"
        if is_labeled:
            json_format = f"{path}/RIP2_{data}-RGB.json"
            with open(json_format, 'r') as file:
                json_data = json.load(file)
        else:
            json_data = None

        rgb = cv2.imread(rgb_format)
        rx = cv2.imread(rx_format)
        id_image = f"bandeja_{data}"

        hiperspectral = envi.open(hdr_format, bil_format)
        h = []
        for i in range(150):
            h.append(hiperspectral.read_band(i))

        info = {
            "rgb": rgb,
            "rx": rx,
            "hiper": np.array(h),
            "json": json_data,
        }

        try:
            pipeline = SegmentationPipeline(info)
            pipeline.run()
            pipeline.save_cropped_seeds(output_dir=DIRETORIO_SAIDA, image_id=id_image)

        except Exception as e:
            print(f"Erro ao processar a {id_image}: {e}")

## Sementes anotadas

In [ ]:
read_seeds(dados_anotados, is_labeled=True)

## Sementes não anotadas

In [ ]:
sementes_nao_anotadas = []
for i in range(1,107):
    num = f"{i:03}"
    if num not in dados_anotados:
        rotulo = num
        sementes_nao_anotadas.append(rotulo)

sementes_nao_anotadas.append("109")
sementes_nao_anotadas.append("130")
sementes_nao_anotadas.append("143")
sementes_nao_anotadas.append("157")

In [ ]:
read_seeds(sementes_nao_anotadas, is_labeled=False)